# 05 — Recommendation System

**Project:** Credit Risk Assessment — *Give Me Some Credit* (Kaggle)

Notebooks 01–04 answered *can we predict default, and can we trust the score?*
`04_evaluation` selected a **calibrated XGBoost** operating at the cost-optimal threshold
$p^{*} \approx 0.167$ and persisted the model with its decision policy. This notebook turns
that artifact into a usable **recommendation system**: a function that takes a *raw*
applicant record — exactly as a loan officer would type it — and returns the three things a
real lending decision requires:

1. a **calibrated probability of default** $\hat{p}$,
2. a **decision** (approve / refer to manual review / decline) under an explicit cost policy,
3. the **reason codes** — the signed, per-applicant factors behind the score, via SHAP.

The third item is not optional polish. Fair-lending regulation (e.g. the U.S. Equal Credit
Opportunity Act, and the GDPR "right to an explanation" in the EU) requires that a declined
applicant be told *why*. A credit model that cannot explain itself is not deployable.

### The inference contract and training/serving skew

The model was trained on *processed* features (cleaned, winsorised, imputed, plus
missingness flags). A new applicant arrives as **raw** values. If the transformation at
inference differs in the slightest from the one at training — a different median, a
different cap, a missed cleaning rule — the model silently receives out-of-distribution
inputs. This failure mode is **training/serving skew**, one of the most common causes of
models that look excellent offline yet misbehave in production.

Notebook 02 persisted the transformed datasets but not a reusable transformer. We therefore
**reconstruct the preprocessing as a single fitted object**, learning its parameters from
the *same* deterministic training split (`random_state=42`), and we **prove faithfulness**
by checking that re-processing the raw test rows reproduces the saved `X_test.csv` exactly.
Only a transformer that passes this test is safe to deploy.

In [1]:
import sys

!{sys.executable} -m pip install xgboost shap --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET = "SeriousDlqin2yrs"
TEST_SIZE = 0.20

DATA_RAW   = Path("..") / "data" / "cs-training.csv"
PROC_DIR   = Path("data") / "processed"
MODELS_DIR = Path("models")

FINAL_MODEL_PATH = MODELS_DIR / "final_model.joblib"
POLICY_PATH      = MODELS_DIR / "decision_policy.json"
XGB_TUNED_PATH   = MODELS_DIR / "xgboost.joblib"

RAW_FEATURES = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]

MISSING_FLAG_COLS = ["MonthlyIncome", "NumberOfDependents"]

PASTDUE_COLS = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

pd.set_option("display.max_columns", None)
print("shap version:", shap.__version__)

shap version: 0.52.0


In [4]:
final_model = joblib.load(FINAL_MODEL_PATH)    
xgb_tuned   = joblib.load(XGB_TUNED_PATH)      

with open(POLICY_PATH, "r", encoding="utf-8") as fh:
    policy = json.load(fh)

DECISION_THRESHOLD = float(policy["threshold"])        
C_FN = float(policy["cost_false_negative"])              
C_FP = float(policy["cost_false_positive"])              

X_test_stored = pd.read_csv(PROC_DIR / "X_test.csv")
STORED_ORDER  = X_test_stored.columns.tolist()          

print(f"Final model        : {type(final_model).__name__}")
print(f"Base model (SHAP)  : {type(xgb_tuned).__name__}")
print(f"Decision policy    : decline-leaning if P(default) > {DECISION_THRESHOLD:.3f}")
print(f"Cost ratio C_FN:C_FP = {C_FN:.0f}:{C_FP:.0f}")
print(f"Stored feature order ({len(STORED_ORDER)}): {STORED_ORDER}")

Final model        : CalibratedClassifierCV
Base model (SHAP)  : XGBClassifier
Decision policy    : decline-leaning if P(default) > 0.167
Cost ratio C_FN:C_FP = 5:1
Stored feature order (12): ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'MonthlyIncome_missing', 'NumberOfDependents_missing']


## 1. The inference pipeline

A deployed prediction is a **composition** of two stages:

$$
\hat{p}(\mathbf{x}_{\text{raw}}) \;=\; \hat{f}_{\text{cal}}\big(T(\mathbf{x}_{\text{raw}})\big),
$$

where $T$ is the preprocessing transform (cleaning, winsorisation, imputation, missingness
flags) and $\hat{f}_{\text{cal}}$ is the calibrated XGBoost from notebook 04. For the result
to be valid, $T$ at serving time must be **identical** to $T$ at training time — same caps,
same medians, same rules. Any divergence is training/serving skew.

$T$ has two kinds of parameters:
- **structural** rules that never change (which codes are junk, which columns get a flag);
- **learned** parameters — the winsor caps and imputation medians — which must be estimated
  on the **training split only**, never on the data we score, to avoid leakage.

The split in `02_preprocessing` is deterministic (`train_test_split(..., random_state=42,
stratify=y)`), so re-running it here recovers the *exact same* raw training rows. We fit $T$
on them, then validate $T$ by reproducing the stored test matrix below.

In [5]:
class CreditPreprocessor:
    """Leakage-safe reconstruction of the 02_preprocessing transform.

    Mirrors notebook 02 exactly: missingness flags on the original values; junk
    codes (96/98) and the impossible age == 0 set to NaN; the two heavy-tailed
    ratios winsorised at train-derived 99th-percentile caps; then median
    imputation of every column that can hold NaN. All learned parameters come
    from the RAW training split only.
    """

    WINSOR_COLS = ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]
    WINSOR_Q = 0.99
    PLACEHOLDERS = [96, 98]                       

    IMPUTE_COLS = ["age", "MonthlyIncome", "NumberOfDependents"] + PASTDUE_COLS

    def __init__(self, stored_order):
        self.stored_order = stored_order
        self.caps_ = {}
        self.medians_ = {}

    def _structural_clean(self, X):
        """Parameter-free rules (02 §4): junk codes and age == 0 become NaN."""
        X = X.copy()
        for c in PASTDUE_COLS:
            X[c] = X[c].mask(X[c].isin(self.PLACEHOLDERS))
        X["age"] = X["age"].mask(X["age"] == 0)
        return X

    def fit(self, X_raw):
        X = self._structural_clean(X_raw[RAW_FEATURES])
        for c in self.WINSOR_COLS:
            self.caps_[c] = float(X[c].quantile(self.WINSOR_Q))
            X[c] = X[c].clip(upper=self.caps_[c])
        for c in self.IMPUTE_COLS:
            self.medians_[c] = float(X[c].median())
        return self

    def transform(self, X_raw):
        X = X_raw[RAW_FEATURES].copy()
        flags = {f"{c}_missing": X[c].isna().astype(int) for c in MISSING_FLAG_COLS}
        X = self._structural_clean(X)
        for c in self.WINSOR_COLS:
            X[c] = X[c].clip(upper=self.caps_[c])
        for c in self.IMPUTE_COLS:
            X[c] = X[c].fillna(self.medians_[c])
        for name, col in flags.items():
            X[name] = col.values
        return X[self.stored_order]

## 2. Proving the transform is faithful

A reconstructed transform is only trustworthy if it reproduces the original. We re-run the
deterministic split to recover the raw test rows, fit the preprocessor on the raw *training*
rows, transform the raw test rows, and assert the result equals the stored `X_test.csv`
**exactly**. If a single cap, median or rule diverged, the assertion fails — turning a silent
production bug into a loud, immediate test failure. This is the engineering guarantee that
makes the system safe to deploy.

In [7]:
raw = pd.read_csv(DATA_RAW)
if "Unnamed: 0" in raw.columns:                 
    raw = raw.drop(columns=["Unnamed: 0"])

X_raw_all, y_all = raw[RAW_FEATURES], raw[TARGET]

X_raw_train, X_raw_test, _, _ = train_test_split(
    X_raw_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)


preprocessor = CreditPreprocessor(stored_order=STORED_ORDER).fit(X_raw_train)
X_test_rebuilt = preprocessor.transform(X_raw_test).reset_index(drop=True)

pd.testing.assert_frame_equal(
    X_test_rebuilt,
    X_test_stored.reset_index(drop=True),
    check_dtype=False,
    atol=1e-6,
)
print(" Faithfulness check passed: the reconstructed transform reproduces X_test.csv.")
print(f"  Learned winsor caps : {preprocessor.caps_}")
print(f"  Learned medians     : {preprocessor.medians_}")

 Faithfulness check passed: the reconstructed transform reproduces X_test.csv.
  Learned winsor caps : {'RevolvingUtilizationOfUnsecuredLines': 1.0944386634299974, 'DebtRatio': 4977.0}
  Learned medians     : {'age': 52.0, 'MonthlyIncome': 5390.0, 'NumberOfDependents': 0.0, 'NumberOfTime30-59DaysPastDueNotWorse': 0.0, 'NumberOfTime60-89DaysPastDueNotWorse': 0.0, 'NumberOfTimes90DaysLate': 0.0}
